# Assignment 1: From Predictions to Profits — Integrative Overbooking Policy Evaluation

## Overview

In the previous two notebooks, you built and evaluated cancellation prediction models (Notebook 1) and explored overbooking decision policies using simulated data (Notebook 2). In this assignment, you will connect these two pieces into a complete pipeline: use a real predictive model trained on real booking data to drive an overbooking policy, then evaluate that policy offline using bootstrap resampling.

The central question is: **does a better prediction model lead to a better overbooking policy?** To answer this, you will start from a baseline model and data pipeline (provided), improve both the features and the model, calibrate the probabilities, construct a constrained overbooking policy, and evaluate whether your improvements translate into higher expected profit and lower tail risk. At the end, you will decompose the dollar value of each improvement — feature engineering, model selection, and calibration — to see which steps matter most for decision quality.

---

## Setup

Run the cell below to import libraries and set global parameters. **Do not modify this cell in your final submission.**

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, ExtraTreesClassifier
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    accuracy_score, roc_auc_score, brier_score_loss, log_loss
)

plt.style.use('fivethirtyeight')
%matplotlib inline

# ── Business parameters (do not change) ─────────────────────────────────────
CAPACITY = 100              # hotel rooms
ROOM_REVENUE = 180          # $ per occupied room
WALK_COST = 1000            # $ per walked guest
CATASTROPHE_THRESHOLD = 5   # walking this many guests is unacceptable
CATASTROPHE_TOL = 0.05      # maximum acceptable P(walked ≥ threshold)
POOL_SIZE = 200             # booking requests per night (hotel is popular)
N_SIM = 10000               # Monte Carlo draws for policy computation
B = 1000                    # bootstrap resamples for evaluation

# ── Master random seed ──────────────────────────────────────────────────────
SEED = 42

---

## Part 1: Data Loading and Baseline Model (15 points)

The code below reproduces the data pipeline from the most popular notebook on Kaggle. It loads the hotel bookings dataset, performs basic preprocessing, and splits into train/test sets. **Read it carefully but do not modify it** — you will build on top of this in Part 2.

In [ ]:
# ── Data loading and preprocessing (provided — do not modify) ───────────────
import kagglehub
data_path = kagglehub.dataset_download('jessemostipak/hotel-booking-demand')
df_raw = pd.read_csv(data_path + '/hotel_bookings.csv')

# Work on a copy; keep df_raw intact in case you need original columns later
df = df_raw.copy()

# Fill missing values
df.fillna(0, inplace=True)

# Drop columns (including leakage-prone ones)
useless_col = [
    'days_in_waiting_list', 'arrival_date_year', 'assigned_room_type',
    'booking_changes', 'reservation_status', 'country', 'days_in_waiting_list'
]
df.drop(useless_col, axis=1, inplace=True)

# Encode categorical variables
cat_cols = [col for col in df.columns if df[col].dtype == 'O']
cat_df = df[cat_cols].copy()

cat_df['reservation_status_date'] = pd.to_datetime(cat_df['reservation_status_date'])
cat_df['year']  = cat_df['reservation_status_date'].dt.year
cat_df['month'] = cat_df['reservation_status_date'].dt.month
cat_df['day']   = cat_df['reservation_status_date'].dt.day
cat_df.drop(['reservation_status_date', 'arrival_date_month'], axis=1, inplace=True)

cat_df['hotel'] = cat_df['hotel'].map({'Resort Hotel': 0, 'City Hotel': 1})
cat_df['meal']  = cat_df['meal'].map({'BB': 0, 'FB': 1, 'HB': 2, 'SC': 3, 'Undefined': 4})
cat_df['market_segment'] = cat_df['market_segment'].map({
    'Direct': 0, 'Corporate': 1, 'Online TA': 2, 'Offline TA/TO': 3,
    'Complementary': 4, 'Groups': 5, 'Undefined': 6, 'Aviation': 7
})
cat_df['distribution_channel'] = cat_df['distribution_channel'].map({
    'Direct': 0, 'Corporate': 1, 'TA/TO': 2, 'Undefined': 3, 'GDS': 4
})
cat_df['reserved_room_type'] = cat_df['reserved_room_type'].map({
    'C': 0, 'A': 1, 'D': 2, 'E': 3, 'G': 4, 'F': 5, 'H': 6, 'L': 7, 'B': 8, 'P': 9
})
cat_df['deposit_type']  = cat_df['deposit_type'].map({'No Deposit': 0, 'Refundable': 1, 'Non Refund': 3})
cat_df['customer_type'] = cat_df['customer_type'].map({'Transient': 0, 'Contract': 1, 'Transient-Party': 2, 'Group': 3})
cat_df['year'] = cat_df['year'].map({2015: 0, 2014: 1, 2016: 2, 2017: 3})

# Remove leakage features
cat_df = cat_df.drop(columns=['year', 'month', 'day'])

# Numerical features with log transform
num_df = df.drop(columns=cat_cols, axis=1)
num_df.drop('is_canceled', axis=1, inplace=True)
for col in ['lead_time', 'arrival_date_week_number', 'arrival_date_day_of_month', 'agent', 'company', 'adr']:
    num_df[col] = np.log(num_df[col] + 1)
num_df['adr'] = num_df['adr'].fillna(value=num_df['adr'].mean())

# Combine and split
X_baseline = pd.concat([cat_df, num_df], axis=1)
y = df['is_canceled']

np.random.seed(SEED)
X_train_base, X_test_base, y_train, y_test = train_test_split(X_baseline, y, test_size=0.30)

# Reset index for clean sampling later
X_train_base = X_train_base.reset_index(drop=True)
X_test_base  = X_test_base.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test  = y_test.reset_index(drop=True)

print(f"Training set: {X_train_base.shape[0]:,} observations, {X_train_base.shape[1]} features")
print(f"Test set:     {X_test_base.shape[0]:,} observations, {X_test_base.shape[1]} features")
print(f"Empirical cancellation rate — train: {y_train.mean():.3f}, test: {y_test.mean():.3f}")
print(f"\nNote: df_raw is preserved with all original columns for feature engineering.")

### 1a. Train and Evaluate a Baseline Model (5 points)

Train a **logistic regression** on the baseline features above. **Store the fitted model as `model_A`.**

Report the following metrics on the **test set**:

| Metric | Value |
|--------|-------|
| AUC | |
| Brier score | |
| Mean predicted P(cancel) | |
| Empirical cancellation rate | |

One of these metrics — the **Brier score** — may be new to you. Look up `sklearn.metrics.brier_score_loss` and answer: what does the Brier score measure, and why might it be more useful than AUC when predicted probabilities will be used to make business decisions? Explain as if you are speaking to a non-technical manager.

💻 **Code required**

✍️ **Written answer** (2–3 sentences on the Brier score)

In [ ]:
# YOUR CODE HERE
# Produce: model_A  (a fitted LogisticRegression)
from sklearn.metrics import roc_auc_score, brier_score_loss

# Train baseline logistic regression
model_A = LogisticRegression(max_iter=2000)
model_A.fit(X_train_base, y_train)

# Predict probabilities on test
probs_A = model_A.predict_proba(X_test_base)[:, 1]

# Metrics
auc = roc_auc_score(y_test, probs_A)
brier = brier_score_loss(y_test, probs_A)
mean_p = probs_A.mean()
emp_rate = float(np.mean(y_test))

pd.DataFrame({
    "Metric": ["AUC", "Brier score", "Mean predicted P(cancel)", "Empirical cancellation rate"],
    "Value":  [auc, brier, mean_p, emp_rate]
})


*Your answer:
The Brier score measures the accuracy of probabilistic predictions: for each example the model outputs a probability p, and the true label is y (y=1 for cancellation, y=0 otherwise). It averages the squared error (p - y)^2 across all examples, so lower means the probabilities are closer to reality. Since we will use p to decide how much to overbook and that directly impacts dollars per night, probability accuracy is especially important.


### 1b. Baseline Calibration (10 points)

Plot a reliability diagram (calibration curve) for `model_A` on the test set. Use 10 bins.

Then answer: Is the baseline model well-calibrated? Where (which probability ranges) does it deviate most from the diagonal?

💻 **Code required**

📊 **Report:** reliability diagram

✍️ **Written answer** (2–3 sentences)

In [ ]:
# YOUR CODE HERE
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
import numpy as np

# Predicted probabilities from model_A on test set
probs_A = model_A.predict_proba(X_test_base)[:, 1]

# 10-bin calibration curve
frac_pos, mean_pred = calibration_curve(
    y_test, probs_A, n_bins=10, strategy="uniform"
)

# Plot reliability diagram
plt.figure(figsize=(6, 6))
plt.plot(mean_pred, frac_pos, marker="o", label="Model A")
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfectly calibrated")
plt.xlabel("Mean predicted probability (bin)")
plt.ylabel("Fraction of positives (empirical)")
plt.title("Reliability Diagram (Model A, 10 bins)")
plt.legend()
plt.grid(True)
plt.show()

# Print bin-wise numbers to help written answer
calib_table = []
for i, (mp, fp) in enumerate(zip(mean_pred, frac_pos), start=1):
    calib_table.append({
        "bin": i,
        "mean_pred_p": float(mp),
        "empirical_rate": float(fp),
        "gap(emp - pred)": float(fp - mp)
    })

pd.DataFrame(calib_table)


*Your answer:* Model A is reasonably calibrated: for low-to-mid probability ranges (roughly 0–0.5), most bins lie close to the diagonal. The largest deviation appears in the high-probability range (about 0.6–0.9), where the empirical cancellation rate is much higher than predicted (e.g., predicted ~0.85 but observed is close to ~0.99), meaning the model tends to underestimate cancellation risk for high-risk bookings. Since overbooking decisions rely on these probabilities, this underestimation could lead to insufficiently conservative decisions in those cases.



---

## Part 2: Improve the Predictive Model (25 points)

In this section, you will try to improve on the baseline through feature engineering, model selection, hyperparameter tuning, and/or calibration. Your improvements must be legitimate — you may not change the outcome variable, leak test information into training, or use `reservation_status` (which reveals the outcome). You may use `df_raw` to access columns that were dropped in the baseline pipeline (e.g., `country`, `arrival_date_year`), as long as the features you create do not leak future information.

You will be evaluated on the **quality and justification** of your choices, not solely on the magnitude of improvement.

**Important:** You must produce four model configurations by the end of this section, as they will all be used in the policy evaluation (Part 4). The four configurations are:

| Label | Features | Model | Calibrated? |
|-------|----------|-------|-------------|
| Config A | Baseline | Logistic Regression | No |
| Config B | Improved | Logistic Regression | No |
| Config C | Improved | Your best model | No |
| Config D | Improved | Your best model | Yes |

Config A is already done (Part 1). You will build B, C, and D below.

### 2a. Feature Engineering (10 points)

Create an improved feature set. You may:
- Add interaction terms or polynomial features
- Create new features from existing columns (e.g., total guests = adults + children + babies)
- Re-introduce columns from `df_raw` that were dropped in the baseline (e.g., `country`, `arrival_date_year`) with appropriate encoding
- Use different encoding strategies for categorical variables
- Apply different transformations to numerical variables
- Add or remove features with a clear rationale

Your improved features must use the **same train/test split indices** (same `y_train`, `y_test`). Store the results as `X_train_improved` and `X_test_improved`.

💻 **Code required**

✍️ **Written answer:** Briefly describe (3–5 sentences) what features you added or changed, and your reasoning.

In [ ]:
# Produce: X_train_improved, X_test_improved  (same rows as X_train_base, X_test_base)

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from scipy.sparse import hstack, csr_matrix

# 1) Start from baseline features (already cleaned/encoded by template)
X_train_imp_df = X_train_base.copy()
X_test_imp_df  = X_test_base.copy()

# 2) Pull the SAME rows from df_raw (indices are aligned in your diagnosis)
df_train = df_raw.loc[X_train_base.index].copy()
df_test  = df_raw.loc[X_test_base.index].copy()

# 3) Engineered numeric features (common in your prediction notebooks, non-leaky)
df_train["total_guests"] = df_train["adults"] + df_train["children"] + df_train["babies"]
df_test["total_guests"]  = df_test["adults"] + df_test["children"] + df_test["babies"]

df_train["total_stay"] = df_train["stays_in_weekend_nights"] + df_train["stays_in_week_nights"]
df_test["total_stay"]  = df_test["stays_in_weekend_nights"] + df_test["stays_in_week_nights"]

df_train["is_family"] = ((df_train["children"] + df_train["babies"]) > 0).astype(int)
df_test["is_family"]  = ((df_test["children"] + df_test["babies"]) > 0).astype(int)

df_train["prev_total"] = df_train["previous_cancellations"] + df_train["previous_bookings_not_canceled"]
df_test["prev_total"]  = df_test["previous_cancellations"] + df_test["previous_bookings_not_canceled"]

# Attach to baseline matrix
for col in ["total_guests", "total_stay", "is_family", "prev_total"]:
    X_train_imp_df[col] = df_train[col].to_numpy()
    X_test_imp_df[col]  = df_test[col].to_numpy()

# Optional: simple interaction features (helps LR a bit, still interpretable)
# Only add if the baseline already has these numeric columns
if "lead_time" in X_train_imp_df.columns:
    X_train_imp_df["lead_time_x_stay"] = X_train_imp_df["lead_time"] * X_train_imp_df["total_stay"]
    X_test_imp_df["lead_time_x_stay"]  = X_test_imp_df["lead_time"] * X_test_imp_df["total_stay"]

if "adr" in X_train_imp_df.columns:
    X_train_imp_df["adr_x_stay"] = X_train_imp_df["adr"] * X_train_imp_df["total_stay"]
    X_test_imp_df["adr_x_stay"]  = X_test_imp_df["adr"] * X_test_imp_df["total_stay"]

# 4) Add a SMALL set of extra categorical features from df_raw (controlled cardinality)
extra_cat = []
for c in ["arrival_date_month", "arrival_date_year", "country", "market_segment", "customer_type"]:
    if c in df_raw.columns:
        extra_cat.append(c)

df_train_extra = df_train[extra_cat].copy() if extra_cat else pd.DataFrame(index=X_train_base.index)
df_test_extra  = df_test[extra_cat].copy()  if extra_cat else pd.DataFrame(index=X_test_base.index)

# Country: top-15 + Other (prevents huge one-hot)
if "country" in df_train_extra.columns:
    top_countries = df_train_extra["country"].value_counts().head(15).index
    df_train_extra["country"] = df_train_extra["country"].where(df_train_extra["country"].isin(top_countries), "Other")
    df_test_extra["country"]  = df_test_extra["country"].where(df_test_extra["country"].isin(top_countries), "Other")

# 5) One-hot encode ONLY these extra categorical columns, then concatenate with baseline+engineered numeric
if len(extra_cat) > 0:
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore"))
    ])
    extra_ohe_train = cat_pipe.fit_transform(df_train_extra)
    extra_ohe_test  = cat_pipe.transform(df_test_extra)

    # numeric part -> sparse
    X_train_num = csr_matrix(X_train_imp_df.to_numpy())
    X_test_num  = csr_matrix(X_test_imp_df.to_numpy())

    X_train_improved = hstack([X_train_num, extra_ohe_train]).tocsr()
    X_test_improved  = hstack([X_test_num, extra_ohe_test]).tocsr()
else:
    X_train_improved = X_train_imp_df.to_numpy()
    X_test_improved  = X_test_imp_df.to_numpy()

# sanity
assert X_train_improved.shape[0] == X_train_base.shape[0]
assert X_test_improved.shape[0] == X_test_base.shape[0]

print("X_train_improved:", X_train_improved.shape)
print("X_test_improved :", X_test_improved.shape)

X_train_improved, X_test_improved

*Your answer:*I improved the feature set by adding a few intuitive engineered features: total_guests (adults + children + babies), total_stay (weekend + weekday nights), and an is_family indicator, since party size and length of stay often relate to cancellation behavior. I also added prev_total (previous cancellations + previous non-canceled bookings) to capture customer history. In addition, I re-introduced time and location signals (arrival year/month and country), while grouping country into top-15 vs “Other” to control cardinality and overfitting. All features are available at decision time and I explicitly removed leakage columns such as reservation_status.



### 2b. Model Selection and Hyperparameter Tuning (10 points)

Choose a model (or a small set of candidates) and tune its hyperparameters. You may use cross-validation, grid search, random search, or any method you prefer. Train your final model on the training set.

You need to produce two models here:
- **`model_B`**: a logistic regression fitted on `X_train_improved`
- **`model_C`**: your best model fitted on `X_train_improved`

Report the following metrics on the test set:

| Metric | Config A (baseline) | Config B | Config C |
|--------|------------|----------|----------|
| AUC | | | |
| Brier score | | | |
| Mean predicted P(cancel) | | | |

💻 **Code required**

📊 **Report:** comparison table

In [ ]:
# YOUR CODE HERE
# Produce: model_B  (LogisticRegression fitted on X_train_improved)
#          model_C  (your best model fitted on X_train_improved)
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, brier_score_loss

# 0) Make sure y is numpy for CV indexing
y_train_arr = y_train.to_numpy() if hasattr(y_train, "to_numpy") else np.asarray(y_train)

# 1) model_B: impute + logistic regression
model_B = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("lr", LogisticRegression(max_iter=3000, solver="liblinear"))
])
model_B.fit(X_train_improved, y_train_arr)

probs_B = model_B.predict_proba(X_test_improved)[:, 1]
auc_B = roc_auc_score(y_test, probs_B)
brier_B = brier_score_loss(y_test, probs_B)
mean_p_B = probs_B.mean()

# 2) model_C: tune C using manual 3-fold CV (still no joblib)
Cs = [0.1, 0.3, 1.0, 3.0, 10.0, 30.0]
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

best_C = None
best_cv_brier = float("inf")

for C in Cs:
    fold_briers = []
    for train_idx, val_idx in cv.split(X_train_improved, y_train_arr):
        X_tr = X_train_improved[train_idx]
        y_tr = y_train_arr[train_idx]
        X_va = X_train_improved[val_idx]
        y_va = y_train_arr[val_idx]

        m = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("lr", LogisticRegression(max_iter=3000, solver="liblinear", C=C))
        ])
        m.fit(X_tr, y_tr)
        p_va = m.predict_proba(X_va)[:, 1]
        fold_briers.append(brier_score_loss(y_va, p_va))

    cv_brier = float(np.mean(fold_briers))
    if cv_brier < best_cv_brier:
        best_cv_brier = cv_brier
        best_C = C

print("Best C selected by 3-fold CV (min Brier):", best_C, "CV Brier:", best_cv_brier)

model_C = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("lr", LogisticRegression(max_iter=3000, solver="liblinear", C=best_C))
])
model_C.fit(X_train_improved, y_train_arr)

probs_C = model_C.predict_proba(X_test_improved)[:, 1]
auc_C = roc_auc_score(y_test, probs_C)
brier_C = brier_score_loss(y_test, probs_C)
mean_p_C = probs_C.mean()

# 3) model_A metrics (baseline) for comparison
probs_A = model_A.predict_proba(X_test_base)[:, 1]
auc_A = roc_auc_score(y_test, probs_A)
brier_A = brier_score_loss(y_test, probs_A)
mean_p_A = probs_A.mean()
emp_rate = float(np.mean(y_test))

comparison = pd.DataFrame({
    "Metric": ["AUC", "Brier score", "Mean predicted P(cancel)", "Empirical cancellation rate"],
    "Config A (baseline)": [auc_A, brier_A, mean_p_A, emp_rate],
    "Config B": [auc_B, brier_B, mean_p_B, emp_rate],
    "Config C": [auc_C, brier_C, mean_p_C, emp_rate],
})
comparison

### 2c. Calibration (5 points)

If `model_C`'s predicted probabilities are not well-calibrated, apply a post-hoc calibration method (Platt scaling or isotonic regression). Plot reliability diagrams **before and after** calibration on the same figure.

If your model is already well-calibrated, show the reliability diagram and explain why no correction is needed.

Store the result as **`model_D`**.

💻 **Code required**

📊 **Report:** reliability diagram(s)

✍️ **Written answer** (2–3 sentences): Did calibration improve the Brier score? Why or why not?

In [ ]:
# YOUR CODE HERE
# Produce: model_D  (calibrated version of model_C, with .predict_proba method)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss

# 1) Calibrate model_C -> model_D (Platt scaling / sigmoid)
# Use 3-fold CV to fit the calibrator on training data
model_D = CalibratedClassifierCV(
    estimator=model_C,
    method="sigmoid",
    cv=3
)
model_D.fit(X_train_improved, y_train)

# 2) Predicted probabilities on test (before vs after)
probs_C = model_C.predict_proba(X_test_improved)[:, 1]
probs_D = model_D.predict_proba(X_test_improved)[:, 1]

# 3) Brier scores
brier_C = brier_score_loss(y_test, probs_C)
brier_D = brier_score_loss(y_test, probs_D)

print("Brier (Config C):", brier_C)
print("Brier (Config D calibrated):", brier_D)

# 4) Reliability diagram (10 bins) on same figure
frac_C, mean_C = calibration_curve(y_test, probs_C, n_bins=10, strategy="uniform")
frac_D, mean_D = calibration_curve(y_test, probs_D, n_bins=10, strategy="uniform")

plt.figure(figsize=(6, 6))
plt.plot(mean_C, frac_C, marker="o", label="Config C (before)")
plt.plot(mean_D, frac_D, marker="o", label="Config D (after calibration)")
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfectly calibrated")
plt.xlabel("Mean predicted probability (bin)")
plt.ylabel("Fraction of positives (empirical)")
plt.title("Reliability Diagram: Before vs After Calibration (10 bins)")
plt.grid(True)
plt.legend()
plt.show()

*Your answer:*I calibrated Config C with sigmoid (Platt scaling) to get Config D. The Brier score didn’t improve and is actually slightly higher, but the difference is tiny. I think this is because Config C was already pretty well-calibrated (the two curves almost overlap and stay close to the diagonal), so calibration doesn’t have much room to help here.


### 2d. Configuration Verification

Run the cell below to verify that all four configurations are ready. This cell should run without error before you proceed.

**Do not modify this cell.**

In [ ]:
# ── Verification cell (do not modify) ────────────────────────────────────────
configs = {
    'A: Baseline + LR':           {'features': 'baseline',  'model_var': 'model_A'},
    'B: Improved + LR':           {'features': 'improved',  'model_var': 'model_B'},
    'C: Improved + Best':         {'features': 'improved',  'model_var': 'model_C'},
    'D: Improved + Best + Calib': {'features': 'improved',  'model_var': 'model_D'},
}

for label, cfg in configs.items():
    m = eval(cfg['model_var'])
    X = X_test_base if cfg['features'] == 'baseline' else X_test_improved
    probs = m.predict_proba(X)[:, 1]
    brier = brier_score_loss(y_test, probs)
    print(f"  {label:<35s}  mean P(cancel) = {probs.mean():.4f}   Brier = {brier:.4f}  ✓")

print("\nAll configurations ready.")

---

## Economics of the Overbooking Decision

Before constructing your policy, review the economic framework from Notebook 2. The key formulas are restated here for reference.

### Business setting

The hotel is popular: every night, more guests want to book than there are rooms. We model this as a **booking queue** of `POOL_SIZE = 200` guests who request a reservation. The hotel **cannot reject** a guest who wants to book — it can only **close the booking channel**. The first 100 guests are always accepted (filling the hotel to capacity). The decision is whether to keep the channel open and accept guest 101, 102, and so on.

### Profit formula

On a given night, the hotel accepts $n$ guests (where $n$ may exceed capacity). Each accepted guest either **shows up** or **cancels/no-shows**. The realized profit is:

$$\pi \;=\; R \times \text{arrivals} \;-\; C \times \text{walked}$$

where:
- $\text{arrivals}$ = number of accepted guests who show up (i.e., `is_canceled == 0`)
- $\text{walked} = \max(0,\; \text{arrivals} - \text{capacity})$
- $R = 180$ (room revenue per arrival)
- $C = 1{,}000$ (cost per walked guest)

A guest who shows up always generates revenue $R$, whether or not they receive a room. A walked guest incurs an *additional* cost $C$ (relocation, compensation, reputation damage). A guest who cancels/no-shows generates zero revenue and zero cost.

### Marginal profit from accepting one more guest

Suppose the hotel has already accepted $n$ guests and is considering accepting one more. We don't yet know this guest's individual cancellation probability, so we use the **average cancellation probability** $\bar{p}$ of the guest population. The expected marginal profit is:

$$E[\Delta\pi] \;=\; (1 - \bar{p}) \;\bigl[\, R \;-\; P(\text{arrivals} \geq \text{capacity} \mid n \text{ accepted}) \cdot C \,\bigr]$$

This is positive as long as:

$$P(\text{arrivals} \geq \text{capacity} \mid n \text{ accepted}) \;<\; \frac{R}{C}$$

With our parameters, $R/C = 180 / 1{,}000 = 0.18$. So the hotel should keep the channel open as long as the probability of being at or over capacity — computed from the cancellation probabilities of the $n$ already-accepted guests — is below 18%.

### Constrained stopping rule

The constrained policy adds a safety condition. Keep the channel open (accept guest $n + 1$) only if **both** conditions hold:

1. **Profit condition:** $P(\text{arrivals} \geq \text{capacity} \mid n \text{ accepted}) < R/C$
2. **Safety condition:** $P(\text{walked} \geq 5 \mid n+1 \text{ accepted}) \leq 0.05$

The hotel closes the channel at the first guest where either condition is violated.

Note that the profit condition looks at the **current** state ($n$ accepted), while the safety condition looks **one step ahead** ($n + 1$ accepted, assuming the next guest has the average cancellation probability $\bar{p}$). This is because the safety condition asks: "if we accept one more, would we still be safe?"

### What uses predictions vs. what uses actual labels

| Step | Uses |
|------|------|
| Computing $P(\text{arrivals} \geq \text{capacity})$ and $P(\text{walked} \geq 5)$ to decide when to stop | Predicted $P(\text{cancel})$ for accepted guests |
| Computing realized profit and walked guests after the night | Actual `is_canceled` labels |

This separation is the core of the predict-then-optimize framework.

---

## Part 3: Construct the Overbooking Policy (20 points)

You will now use your calibrated model to construct an overbooking decision rule following the sequential booking framework from Notebook 2.

**Setup for each simulated night:**
- Sample `POOL_SIZE = 200` guests from the test set as the booking queue for the night
- For each guest, you observe their predicted $P(\text{cancel})$ from your model and their actual `is_canceled` label
- The first 100 guests are always accepted (the hotel fills to capacity)
- Starting from guest 101, the policy decides after each acceptance whether to keep the channel open or close it
- The **stopping decision** uses predicted probabilities; the **outcome** uses actual labels

Before writing the policy functions, compute the predicted probabilities you will need. The cell below should be filled in after completing Part 2.

### 3.0 Prepare Predicted Probabilities

Compute predicted cancellation probabilities on the test set for each of your four model configurations. Store them as a dictionary for easy access later.

Also compute the average cancellation probability $\bar{p}$ from the test set, which is used in the marginal profit formula for the unknown next guest.

💻 **Code required**

In [ ]:
# YOUR CODE HERE
# Produce: probs_dict, a dictionary with keys 'A', 'B', 'C', 'D'
#   probs_dict['A'] = model_A.predict_proba(X_test_base)[:, 1]       # array of length len(y_test)
#   probs_dict['B'] = model_B.predict_proba(X_test_improved)[:, 1]
#   probs_dict['C'] = model_C.predict_proba(X_test_improved)[:, 1]
#   probs_dict['D'] = model_D.predict_proba(X_test_improved)[:, 1]
#
# Also produce: p_bar = y_test.mean()   (average cancellation rate in the test population)
# Produce: probs_dict with keys 'A','B','C','D' and p_bar

import numpy as np

probs_dict = {}

# A uses baseline features
probs_dict["A"] = model_A.predict_proba(X_test_base)[:, 1]

# B/C/D use improved features
probs_dict["B"] = model_B.predict_proba(X_test_improved)[:, 1]
probs_dict["C"] = model_C.predict_proba(X_test_improved)[:, 1]
probs_dict["D"] = model_D.predict_proba(X_test_improved)[:, 1]

# Average cancellation probability in the test population
p_bar = float(y_test.mean())

# quick sanity checks
for k in ["A","B","C","D"]:
    arr = probs_dict[k]
    print(k, "len=", len(arr), "mean=", float(np.mean(arr)))

print("p_bar =", p_bar)


### 3a. Sampling and Outcome Functions (5 points)

Write two functions:

**1. `sample_night(y_test, probs_test, rng, pool_size=POOL_SIZE)`**

Sample `pool_size` guests (without replacement) from the test set to form the booking queue for one night. Returns:
- `y_queue`: array of actual `is_canceled` labels for the sampled guests (length `pool_size`)
- `probs_queue`: array of predicted $P(\text{cancel})$ for the sampled guests (length `pool_size`)

The `rng` argument is a `np.random.RandomState` object passed in from the caller. **Do not create new random states inside the function.**

**2. `compute_outcome(y_queue, n_accepted, capacity=CAPACITY)`**

Given the actual cancellation labels for the full queue and the number of guests accepted ($n$), compute the realized outcome:
- `arrivals` = number of the first $n$ guests whose `is_canceled == 0`
- `walked` = $\max(0,\; \text{arrivals} - \text{capacity})$
- `profit` = $R \times \text{arrivals} - C \times \text{walked}$

Returns a dictionary with keys `'n_accepted'`, `'arrivals'`, `'walked'`, `'profit'`.

**Verify** your functions: sample one night using `rng = np.random.RandomState(SEED)` and `probs_dict['D']`, then compute outcomes for $n = 100, 105, 110, 115, 120$.

💻 **Code required**

📊 **Report:** table of (n_accepted, arrivals, walked, profit) for one sampled night

In [ ]:
import numpy as np
import pandas as pd

R = 180  # room revenue per arrival (given in prompt)

def sample_night(y_test, probs_test, rng, pool_size=POOL_SIZE):
    idx = rng.choice(len(y_test), size=pool_size, replace=False)
    y_arr = y_test.to_numpy() if hasattr(y_test, "to_numpy") else np.asarray(y_test)
    return y_arr[idx], np.asarray(probs_test)[idx]

def compute_outcome(y_queue, n_accepted, capacity=CAPACITY):
    y_queue = np.asarray(y_queue)
    accepted = y_queue[:n_accepted]
    arrivals = int(np.sum(accepted == 0))
    walked = int(max(0, arrivals - capacity))
    profit = float(R * arrivals - WALK_COST * walked)
    return {"n_accepted": int(n_accepted), "arrivals": arrivals, "walked": walked, "profit": profit}

rng = np.random.RandomState(SEED)
y_queue, probs_queue = sample_night(y_test, probs_dict["D"], rng, pool_size=POOL_SIZE)

rows = [compute_outcome(y_queue, n, capacity=CAPACITY) for n in [100, 105, 110, 115, 120]]
report_df = pd.DataFrame(rows)
report_df

### 3b. Probability Computation via Monte Carlo (5 points)

Write a function that estimates arrival probabilities using Monte Carlo simulation, following the approach from Notebook 2:

**`mc_p_arrivals_geq(cancel_probs, threshold, rng, n_sim=N_SIM)`**

Given a vector of predicted cancellation probabilities for the currently accepted guests, estimate $P(\text{arrivals} \geq \text{threshold})$:
- For each of `n_sim` draws, simulate independent cancellations: guest $i$ cancels with probability `cancel_probs[i]`
- Count the number of arrivals (guests who do not cancel)
- Return the fraction of draws where arrivals $\geq$ threshold

This single function is sufficient to check both the profit condition ($P(\text{arrivals} \geq \text{capacity})$) and the safety condition ($P(\text{arrivals} \geq \text{capacity} + \text{threshold})$) by passing different values of `threshold`.

💻 **Code required**

In [ ]:
# YOUR CODE HERE
import numpy as np

def mc_p_arrivals_geq(cancel_probs, threshold, rng, n_sim=N_SIM):
    """
    Monte Carlo estimate of P(arrivals >= threshold), where arrivals are non-cancellations.

    cancel_probs: array-like of length n (P(cancel) for each accepted guest)
    threshold: int, the arrivals threshold
    rng: np.random.RandomState (passed from caller)
    n_sim: number of Monte Carlo simulations
    """
    cancel_probs = np.asarray(cancel_probs, dtype=float)
    n = cancel_probs.shape[0]
    if n == 0:
        return 0.0

    # Simulate cancellations: U < p_cancel => cancel (True)
    U = rng.rand(n_sim, n)
    cancels = (U < cancel_probs)

    # arrivals = number of non-cancels
    arrivals = (~cancels).sum(axis=1)

    return float(np.mean(arrivals >= threshold))

### 3c. Constrained Stopping Policy (10 points)

Write a function that implements the constrained stopping rule:

**`decide_n_accepted(probs_queue, p_bar, rng)`**

Given the predicted cancellation probabilities for all guests in the queue, decide how many to accept:

1. Always accept the first `CAPACITY` (= 100) guests.
2. After accepting guest $n$ (starting from $n = 100$), decide whether to **keep the channel open** and accept guest $n + 1$. Check **both** conditions using `mc_p_arrivals_geq` on the cancellation probabilities of the $n$ currently accepted guests:
   - **Profit condition:** $P(\text{arrivals} \geq \text{capacity} \mid n \text{ accepted}) < R / C$
   - **Safety condition (look-ahead):** Temporarily append $\bar{p}$ to the list of $n$ accepted guests' probabilities to represent the unknown next guest. Check that $P(\text{arrivals} \geq \text{capacity} + \text{CATASTROPHE\_THRESHOLD} \mid n+1 \text{ guests}) \leq \text{CATASTROPHE\_TOL}$
3. If both conditions hold, accept guest $n + 1$ and repeat. If either condition fails, **close the channel** — do not accept guest $n + 1$. Return $n$.

Cap at `len(probs_queue)` (the queue may run out before the policy stops, though with `POOL_SIZE = 200` this is unlikely).

In pseudocode:
```
n = CAPACITY                           # accept first 100 guests
while n < len(probs_queue):
    probs_accepted = probs_queue[:n]    # probabilities of the n accepted guests
    if P(arrivals ≥ CAPACITY | probs_accepted) ≥ R/C:
        break                           # profit condition fails → close
    probs_lookahead = append p_bar to probs_accepted
    if P(arrivals ≥ CAPACITY + CATASTROPHE_THRESHOLD | probs_lookahead) > CATASTROPHE_TOL:
        break                           # safety condition fails → close
    n += 1                              # accept next guest
return n
```

**Test** your policy on 5 sampled nights using `rng = np.random.RandomState(SEED)` and `probs_dict['D']`. For each night, report the number accepted, the estimated $P(\text{over capacity})$ at that point, and the estimated $P(\text{catastrophe})$ at that point.

💻 **Code required**

📊 **Report:** table of results for 5 nights

✍️ **Written answer** (2–3 sentences): Does the number accepted vary across nights? What drives the variation?

In [ ]:
# YOUR CODE HERE
import numpy as np
import pandas as pd

R = 180  # given in prompt

def decide_n_accepted(probs_queue, p_bar, rng):
    probs_queue = np.asarray(probs_queue, dtype=float)
    n = CAPACITY  # always accept first 100

    while n < len(probs_queue):
        probs_accepted = probs_queue[:n]

        # Profit condition: P(arrivals >= capacity | n accepted) < R / WALK_COST
        p_overcap = mc_p_arrivals_geq(probs_accepted, threshold=CAPACITY, rng=rng, n_sim=N_SIM)
        if p_overcap >= (R / WALK_COST):
            break

        # Safety condition (look-ahead): add average cancel prob for unknown next guest
        probs_lookahead = np.append(probs_accepted, p_bar)
        p_catastrophe = mc_p_arrivals_geq(
            probs_lookahead,
            threshold=CAPACITY + CATASTROPHE_THRESHOLD,
            rng=rng,
            n_sim=N_SIM
        )
        if p_catastrophe > CATASTROPHE_TOL:
            break

        n += 1

    return int(n)

# ----------------------------
# Test policy on 5 sampled nights
# ----------------------------
rng = np.random.RandomState(SEED)

rows = []
for night in range(1, 6):
    y_queue, probs_queue = sample_night(y_test, probs_dict["D"], rng, pool_size=POOL_SIZE)
    n_acc = decide_n_accepted(probs_queue, p_bar, rng)

    probs_acc = probs_queue[:n_acc]
    p_overcap = mc_p_arrivals_geq(probs_acc, threshold=CAPACITY, rng=rng, n_sim=N_SIM)

    probs_look = np.append(probs_acc, p_bar)
    p_cat = mc_p_arrivals_geq(probs_look, threshold=CAPACITY + CATASTROPHE_THRESHOLD, rng=rng, n_sim=N_SIM)

    rows.append({
        "night": night,
        "n_accepted": n_acc,
        "P(over capacity)": p_overcap,
        "P(catastrophe)": p_cat
    })

pd.DataFrame(rows)

*Your answer:*Yes, it varies. In my 5-night test, n_accepted ranges roughly from 139 to 157, which means the sampled guests differ night to night. The variation is mainly driven by the distribution of predicted cancellation probabilities: when more cancellations are expected, the policy accepts more to avoid empty rooms; when many guests are likely to show up, the over-capacity (or safety) probability reaches the thresholds sooner, so the policy stops earlier.





---

## Part 4: Offline Policy Evaluation (25 points)

You will now evaluate your ML-based policy against two baselines using bootstrap resampling. In each bootstrap iteration:

1. Sample a night (a queue of 200 guests) from the test data
2. Apply each policy to determine how many guests to accept
3. Compute the realized outcome using **actual labels**

The three policies to compare:

| Policy | Description |
|--------|-------------|
| **No overbooking** | Accept exactly 100 guests (capacity) |
| **Fixed overbook** | Accept exactly 130 guests |
| **ML policy (Config D)** | Constrained stopping rule from Part 3c, using Config D probabilities |

Use your Monte Carlo `decide_n_accepted` from Part 3c. (If it is too slow for 1,000 iterations, you may implement a normal approximation — see bonus question 4e.)

### 4a. Run Bootstrap Evaluation (5 points)

Write a bootstrap loop that evaluates all three policies over $B = 1{,}000$ resampled nights. For each policy and each night, store: profit, number of walked guests, and number of guests accepted.

**Seed handling:** Initialize `rng = np.random.RandomState(SEED)` before the loop. Use this single `rng` for all sampling and decision calls throughout the loop, so the sequence is reproducible.

💻 **Code required**

In [ ]:
import numpy as np
import pandas as pd
from math import erf

R = 180  # from prompt

def normal_p_arrivals_geq(cancel_probs, threshold):
    cancel_probs = np.asarray(cancel_probs, dtype=float)
    if cancel_probs.size == 0:
        return 0.0
    p_show = 1.0 - cancel_probs
    mu = p_show.sum()
    var = (p_show * (1 - p_show)).sum()
    sigma = np.sqrt(max(var, 1e-9))

    # continuity correction
    z = (threshold - 0.5 - mu) / sigma
    Phi = 0.5 * (1.0 + erf(z / np.sqrt(2.0)))
    return float(1.0 - Phi)

def decide_n_accepted_fast(probs_queue, p_bar):
    probs_queue = np.asarray(probs_queue, dtype=float)
    n = CAPACITY
    while n < len(probs_queue):
        probs_accepted = probs_queue[:n]

        p_overcap = normal_p_arrivals_geq(probs_accepted, threshold=CAPACITY)
        if p_overcap >= (R / WALK_COST):
            break

        probs_lookahead = np.append(probs_accepted, p_bar)
        p_cat = normal_p_arrivals_geq(
            probs_lookahead,
            threshold=CAPACITY + CATASTROPHE_THRESHOLD
        )
        if p_cat > CATASTROPHE_TOL:
            break

        n += 1
    return int(n)

B = 1000
rng = np.random.RandomState(SEED)

records = []
for b in range(B):
    y_queue, probs_queue = sample_night(y_test, probs_dict["D"], rng, pool_size=POOL_SIZE)

    out_no = compute_outcome(y_queue, n_accepted=CAPACITY, capacity=CAPACITY)
    records.append({"b": b, "policy": "no_overbooking", **out_no})

    out_fixed = compute_outcome(y_queue, n_accepted=130, capacity=CAPACITY)
    records.append({"b": b, "policy": "fixed_130", **out_fixed})

    n_ml = decide_n_accepted_fast(probs_queue, p_bar)
    out_ml = compute_outcome(y_queue, n_accepted=n_ml, capacity=CAPACITY)
    records.append({"b": b, "policy": "ml_policy_D", **out_ml})

results_4a = pd.DataFrame(records)
results_4a.head()

### 4b. Summary Statistics (5 points)

Report the following table:

| Metric | No Overbooking | Fixed (n=130) | ML Policy |
|--------|---------------|-------------|-----------|
| Expected profit per night | | | |
| Std. dev. of profit | | | |
| P(walked ≥ 1) | | | |
| P(walked ≥ 5) | | | |
| Average n accepted | | | |
| Worst-case walked | | | |

Also report:
- **Win rate**: fraction of nights where the ML policy profit exceeds the fixed-overbook baseline
- **95% confidence interval** for the profit difference (ML minus fixed-overbook), computed as the 2.5th and 97.5th percentiles of the per-night profit differences

💻 **Code required**

📊 **Report:** summary table and CI

In [ ]:
# YOUR CODE HERE
import numpy as np
import pandas as pd

# Pivot to have one row per night, one column per policy
pivot_profit = results_4a.pivot(index="b", columns="policy", values="profit")
pivot_walked = results_4a.pivot(index="b", columns="policy", values="walked")
pivot_acc    = results_4a.pivot(index="b", columns="policy", values="n_accepted")

# Ensure the three policies exist (names from your results_4a)
pol_no = "no_overbooking"
pol_fx = "fixed_130"
pol_ml = "ml_policy_D"

# Summary metrics
expected_profit = {
    pol_no: float(pivot_profit[pol_no].mean()),
    pol_fx: float(pivot_profit[pol_fx].mean()),
    pol_ml: float(pivot_profit[pol_ml].mean()),
}

std_profit = {
    pol_no: float(pivot_profit[pol_no].std(ddof=1)),
    pol_fx: float(pivot_profit[pol_fx].std(ddof=1)),
    pol_ml: float(pivot_profit[pol_ml].std(ddof=1)),
}

p_walked_ge1 = {
    pol_no: float((pivot_walked[pol_no] >= 1).mean()),
    pol_fx: float((pivot_walked[pol_fx] >= 1).mean()),
    pol_ml: float((pivot_walked[pol_ml] >= 1).mean()),
}

p_walked_ge5 = {
    pol_no: float((pivot_walked[pol_no] >= 5).mean()),
    pol_fx: float((pivot_walked[pol_fx] >= 5).mean()),
    pol_ml: float((pivot_walked[pol_ml] >= 5).mean()),
}

avg_accepted = {
    pol_no: float(pivot_acc[pol_no].mean()),
    pol_fx: float(pivot_acc[pol_fx].mean()),
    pol_ml: float(pivot_acc[pol_ml].mean()),
}

worst_walked = {
    pol_no: int(pivot_walked[pol_no].max()),
    pol_fx: int(pivot_walked[pol_fx].max()),
    pol_ml: int(pivot_walked[pol_ml].max()),
}

summary_table = pd.DataFrame({
    "Metric": [
        "Expected profit per night",
        "Std. dev. of profit",
        "P(walked >= 1)",
        "P(walked >= 5)",
        "Average n accepted",
        "Worst-case walked",
    ],
    "No Overbooking": [
        expected_profit[pol_no],
        std_profit[pol_no],
        p_walked_ge1[pol_no],
        p_walked_ge5[pol_no],
        avg_accepted[pol_no],
        worst_walked[pol_no],
    ],
    "Fixed (n=130)": [
        expected_profit[pol_fx],
        std_profit[pol_fx],
        p_walked_ge1[pol_fx],
        p_walked_ge5[pol_fx],
        avg_accepted[pol_fx],
        worst_walked[pol_fx],
    ],
    "ML Policy": [
        expected_profit[pol_ml],
        std_profit[pol_ml],
        p_walked_ge1[pol_ml],
        p_walked_ge5[pol_ml],
        avg_accepted[pol_ml],
        worst_walked[pol_ml],
    ],
})

summary_table

In [ ]:
diff = pivot_profit[pol_ml] - pivot_profit[pol_fx]

win_rate = float((diff > 0).mean())
ci_low = float(np.percentile(diff, 2.5))
ci_high = float(np.percentile(diff, 97.5))

print("Win rate (ML profit > fixed_130):", win_rate)
print("95% CI for per-night profit difference (ML - fixed_130):", (ci_low, ci_high))

### 4c. Visualize Policy Comparison (5 points)

Produce **two** plots:

1. **Profit distributions**: overlaid histograms or KDE plots for all three policies
2. **Walked-guest distributions**: overlaid histograms for all three policies, with a vertical line at the catastrophe threshold (5 guests)

💻 **Code required**

📊 **Report:** two figures

In [ ]:
# YOUR CODE HERE
import matplotlib.pyplot as plt

policies = ["no_overbooking", "fixed_130", "ml_policy_D"]

# 1) Profit distributions (overlaid histograms)
plt.figure(figsize=(8, 5))
for p in policies:
    vals = results_4a.loc[results_4a["policy"] == p, "profit"].to_numpy()
    plt.hist(vals, bins=30, alpha=0.5, density=True, label=p)
plt.xlabel("Profit per night")
plt.ylabel("Density")
plt.title("Profit Distributions Across Policies")
plt.legend()
plt.grid(True)
plt.show()

# 2) Walked-guest distributions (overlaid histograms) + catastrophe line
plt.figure(figsize=(8, 5))
# choose bins to make walked discrete-looking
max_walk = int(results_4a["walked"].max())
bins = np.arange(-0.5, max_walk + 1.5, 1)

for p in policies:
    vals = results_4a.loc[results_4a["policy"] == p, "walked"].to_numpy()
    plt.hist(vals, bins=bins, alpha=0.5, density=True, label=p)

plt.axvline(CATASTROPHE_THRESHOLD, linestyle="--", label=f"catastrophe={CATASTROPHE_THRESHOLD}")
plt.xlabel("Number of walked guests")
plt.ylabel("Density")
plt.title("Walked-Guest Distributions Across Policies")
plt.legend()
plt.grid(True)
plt.show()

### 4d. Dollar Value of Each ML Component (10 points)

Now evaluate how each improvement in the ML pipeline translates into decision quality. Run the bootstrap evaluation **four times**, once for each model configuration. In each run, the stopping policy uses the same `decide_n_accepted` logic but with different predicted probabilities:

| Run | Config | Predicted probabilities from |
|-----|--------|------------------------------|
| 1 | A | `probs_dict['A']` (baseline features + LR) |
| 2 | B | `probs_dict['B']` (improved features + LR) |
| 3 | C | `probs_dict['C']` (improved features + best model) |
| 4 | D | `probs_dict['D']` (improved features + best model + calibration) |

**Seed handling for paired comparison:** Reset `rng = np.random.RandomState(SEED)` at the start of **each** of the four runs. Because the same seed produces the same sequence of sampled guest indices, all four configurations face exactly the same 1,000 nights. This ensures a clean paired comparison: any difference in profit is due to the policy, not the guests.

Report the decomposition table:

| Component | Config | E[Profit]/night | Incremental $/night |
|-----------|--------|-----------------|---------------------|
| Baseline | A | | — |
| + Feature engineering | B | | E[B] − E[A] |
| + Model selection | C | | E[C] − E[B] |
| + Calibration | D | | E[D] − E[C] |
| **Total improvement** | | | E[D] − E[A] |

Use `%%time` at the top of your code cell so you can compare wall-clock time with the bonus question (4e).

💻 **Code required**

📊 **Report:** decomposition table

✍️ **Written answer** (3–5 sentences): Which component contributed the most dollar value? Does this surprise you? How does this relate to the Brier score improvements you observed in Part 2?

In [ ]:
%%time
import numpy as np
import pandas as pd
from math import erf

R = 180  # from prompt
B = 1000

def normal_p_arrivals_geq(cancel_probs, threshold):
    cancel_probs = np.asarray(cancel_probs, dtype=float)
    if cancel_probs.size == 0:
        return 0.0
    p_show = 1.0 - cancel_probs
    mu = p_show.sum()
    var = (p_show * (1 - p_show)).sum()
    sigma = np.sqrt(max(var, 1e-9))
    z = (threshold - 0.5 - mu) / sigma
    Phi = 0.5 * (1.0 + erf(z / np.sqrt(2.0)))
    return float(1.0 - Phi)

def decide_n_accepted_fast(probs_queue, p_bar):
    probs_queue = np.asarray(probs_queue, dtype=float)
    n = CAPACITY
    while n < len(probs_queue):
        probs_accepted = probs_queue[:n]

        p_overcap = normal_p_arrivals_geq(probs_accepted, threshold=CAPACITY)
        if p_overcap >= (R / WALK_COST):
            break

        probs_lookahead = np.append(probs_accepted, p_bar)
        p_cat = normal_p_arrivals_geq(probs_lookahead, threshold=CAPACITY + CATASTROPHE_THRESHOLD)
        if p_cat > CATASTROPHE_TOL:
            break

        n += 1
    return int(n)

y_arr = y_test.to_numpy() if hasattr(y_test, "to_numpy") else np.asarray(y_test)

def eval_config(config_label):
    rng = np.random.RandomState(SEED)  # reset for paired comparison
    profits = []

    for _ in range(B):
        idx = rng.choice(len(y_arr), size=POOL_SIZE, replace=False)
        y_queue = y_arr[idx]
        probs_queue = probs_dict[config_label][idx]

        n_acc = decide_n_accepted_fast(probs_queue, p_bar)
        out = compute_outcome(y_queue, n_accepted=n_acc, capacity=CAPACITY)
        profits.append(float(out["profit"]))

    return float(np.mean(profits))

E = {cfg: eval_config(cfg) for cfg in ["A", "B", "C", "D"]}

decomp = pd.DataFrame([
    {"Component": "Baseline", "Config": "A", "E[Profit]/night": E["A"], "Incremental $/night": np.nan},
    {"Component": "+ Feature engineering", "Config": "B", "E[Profit]/night": E["B"], "Incremental $/night": E["B"] - E["A"]},
    {"Component": "+ Model selection", "Config": "C", "E[Profit]/night": E["C"], "Incremental $/night": E["C"] - E["B"]},
    {"Component": "+ Calibration", "Config": "D", "E[Profit]/night": E["D"], "Incremental $/night": E["D"] - E["C"]},
    {"Component": "Total improvement", "Config": "D - A", "E[Profit]/night": E["D"], "Incremental $/night": E["D"] - E["A"]},
])

decomp

*Your answer:*From the decomposition table, the dollar impacts of each ML component are very small (only a few dollars per night), and overall D is slightly lower than A (about -$3.2/night). The largest positive component is model selection (C vs B is about +$1.4/night), but the magnitude is still tiny, while feature engineering and calibration are slightly negative here. This makes sense because the Brier score improvements in Part 2 were minimal, so the predicted probabilities are already very similar across configs; with a threshold-based stopping rule, small probability changes do not consistently shift decisions enough to create large profit gains.



### 4e. Speed Comparison: Normal Approximation (5 bonus points)

The Monte Carlo approach for `decide_n_accepted` may have been slow during the bootstrap evaluation. An alternative is the **normal approximation**: if guests cancel independently with probabilities $p_1, \ldots, p_n$, then the number of arrivals is approximately normal:

$$\text{arrivals} \;\approx\; Normal \!\left(\sum_i (1 - p_i),\;\sum_i p_i(1 - p_i)\right)$$

so $P(\text{arrivals} \geq t)$ can be computed in closed form via `scipy.stats.norm` (with continuity correction: use $t - 0.5$ instead of $t$).

Do the following:

1. Implement `normal_p_arrivals_geq(cancel_probs, threshold)` and a corresponding `decide_n_accepted_normal(probs_queue, p_bar)` that replaces all MC calls with the normal approximation.
2. Re-run the **dollar-value decomposition from 4d** using the normal approximation. Use `%%time` at the top of the cell to report wall-clock time.
3. Report a comparison table:

| Config | E[Profit] (MC) | E[Profit] (Normal) | Difference |
|--------|---------------|-------------------|------------|
| A | | | |
| B | | | |
| C | | | |
| D | | | |

Also report:
- Wall-clock time for the MC version (from 4d) vs. the normal version (this cell)

💻 **Code required**

📊 **Report:** comparison table and timing

✍️ **Written answer** (2–3 sentences): Is the profit difference between MC and normal approximation economically meaningful? When would you prefer one over the other?

In [ ]:
%%time
import numpy as np
import pandas as pd
from math import erf

R = 180

B_NORMAL = 1000   # normal approximation: fast, keep 1000
B_MC = 200        # MC: smaller to finish quickly
N_SIM_MC = 800    # MC sims per probability call (keep modest)

y_arr = y_test.to_numpy() if hasattr(y_test, "to_numpy") else np.asarray(y_test)

def normal_p_arrivals_geq(cancel_probs, threshold):
    cancel_probs = np.asarray(cancel_probs, dtype=float)
    if cancel_probs.size == 0:
        return 0.0
    p_show = 1.0 - cancel_probs
    mu = p_show.sum()
    var = (p_show * (1 - p_show)).sum()
    sigma = np.sqrt(max(var, 1e-9))
    z = (threshold - 0.5 - mu) / sigma
    Phi = 0.5 * (1.0 + erf(z / np.sqrt(2.0)))
    return float(1.0 - Phi)

def decide_n_accepted_normal(probs_queue, p_bar):
    probs_queue = np.asarray(probs_queue, dtype=float)
    n = CAPACITY
    while n < len(probs_queue):
        probs_accepted = probs_queue[:n]
        p_overcap = normal_p_arrivals_geq(probs_accepted, threshold=CAPACITY)
        if p_overcap >= (R / WALK_COST):
            break
        probs_look = np.append(probs_accepted, p_bar)
        p_cat = normal_p_arrivals_geq(probs_look, threshold=CAPACITY + CATASTROPHE_THRESHOLD)
        if p_cat > CATASTROPHE_TOL:
            break
        n += 1
    return int(n)

def mc_p_arrivals_geq(cancel_probs, threshold, rng, n_sim):
    cancel_probs = np.asarray(cancel_probs, dtype=float)
    n = cancel_probs.size
    if n == 0:
        return 0.0
    U = rng.rand(n_sim, n)
    cancels = (U < cancel_probs)
    arrivals = (~cancels).sum(axis=1)
    return float(np.mean(arrivals >= threshold))

def decide_n_accepted_mc(probs_queue, p_bar, rng, n_sim):
    probs_queue = np.asarray(probs_queue, dtype=float)
    n = CAPACITY
    while n < len(probs_queue):
        probs_accepted = probs_queue[:n]
        p_overcap = mc_p_arrivals_geq(probs_accepted, threshold=CAPACITY, rng=rng, n_sim=n_sim)
        if p_overcap >= (R / WALK_COST):
            break
        probs_look = np.append(probs_accepted, p_bar)
        p_cat = mc_p_arrivals_geq(probs_look, threshold=CAPACITY + CATASTROPHE_THRESHOLD, rng=rng, n_sim=n_sim)
        if p_cat > CATASTROPHE_TOL:
            break
        n += 1
    return int(n)

def eval_config(config_label, mode="normal"):
    if mode == "normal":
        B = B_NORMAL
    else:
        B = B_MC

    rng = np.random.RandomState(SEED)  # paired within each mode
    profits = []
    for _ in range(B):
        idx = rng.choice(len(y_arr), size=POOL_SIZE, replace=False)
        y_queue = y_arr[idx]
        probs_queue = probs_dict[config_label][idx]

        if mode == "normal":
            n_acc = decide_n_accepted_normal(probs_queue, p_bar)
        else:
            n_acc = decide_n_accepted_mc(probs_queue, p_bar, rng=rng, n_sim=N_SIM_MC)

        out = compute_outcome(y_queue, n_accepted=n_acc, capacity=CAPACITY)
        profits.append(float(out["profit"]))
    return float(np.mean(profits))

E_normal = {cfg: eval_config(cfg, mode="normal") for cfg in ["A","B","C","D"]}
E_mc     = {cfg: eval_config(cfg, mode="mc")     for cfg in ["A","B","C","D"]}

comp = pd.DataFrame([
    {"Config": cfg,
     "E[Profit] (MC, B=%d, N_SIM=%d)" % (B_MC, N_SIM_MC): E_mc[cfg],
     "E[Profit] (Normal, B=%d)" % B_NORMAL: E_normal[cfg],
     "Difference (Normal - MC)": E_normal[cfg] - E_mc[cfg]}
    for cfg in ["A","B","C","D"]
])

comp

*Your answer:*
The normal approximation and Monte Carlo estimates are very close: the differences are on the order of a few tens of dollars per night (about -53 to +73 here), which is not economically meaningful compared to ~17k dollars/night. Normal is much faster and is suitable for large bootstrap loops, while MC is more faithful and preferable when tail risk is critical or the distribution is not well-approximated by a normal. In practice, I would iterate with Normal and validate with higher-N_SIM MC at the end.


---

## Part 5: Reflection (15 points)

Answer each question in 3–5 sentences.

### 5a. Prediction Quality vs. Decision Quality (5 points)

Compare the Brier score improvements across your four configurations (A → B → C → D) with the corresponding profit improvements from Part 4d. Is the relationship between prediction quality and decision quality linear, concave, or something else? Give an intuitive explanation for the pattern you observe.

✍️ **Written answer**

*Your answer:*Across A→B→C→D, the Brier score changes are very small, and the corresponding profit differences in Part 4d are also close to zero (D−A is only a few dollars per night). This suggests the relationship between prediction quality and decision quality is not linear; it is more threshold-based/piecewise because the stopping policy changes decisions only when key probabilities cross decision thresholds (e.g., P(over capacity) around 0.18 and P(catastrophe) around 0.05). Since the probability improvements here were tiny and did not consistently shift those threshold comparisons, the decisions (n_accepted) and profits barely changed. In other words, small Brier improvements may not translate into dollars unless they affect the critical decision boundary.



### 5b. Assumptions and Validity (5 points)

Your bootstrap evaluation resamples guests from the test set and treats them as representative of future nights. Give one specific, realistic scenario in which this assumption would fail. Explain the direction of the bias: would your evaluation overstate or understate the true policy performance?

✍️ **Written answer**

*Your answer:*A realistic failure case is distribution shift: for example, the hotel changes cancellation/deposit policies or there is a major event/transport disruption, so the future guest mix and cancellation rates differ from the test set. Our bootstrap assumes future nights look like random samples from the test data, which breaks under such shifts. If the true future cancellation rate is lower (more guests show up), we would underestimate over-capacity risk and therefore overestimate policy profit (more walked guests and higher costs in reality). If cancellations become higher, we may underestimate profit and make the policy look overly conservative.



### 5c. Catastrophe Constraints in Practice (5 points)

The catastrophe threshold (walking 5+ guests) and tolerance (5% probability) were set by the assignment. In a real hotel, who should set these parameters, and what information would they need? Would you expect these values to be the same for a luxury resort and a budget airport hotel?

✍️ **Written answer**

*Your answer:*In practice, these parameters should be set jointly by revenue management/operations and brand-risk leadership (e.g., Revenue Manager, Operations team, GM) because they trade off revenue, customer experience, and reputational risk. They would need inputs such as the direct cost of walking (compensation, relocation, rate differences), the impact on ratings/loyalty and long-term revenue, and historical complaint rates from walk events, along with seasonality and channel mix. I would not expect the same thresholds for a luxury resort versus a budget airport hotel: luxury properties typically adopt stricter thresholds/lower tolerance due to brand and service expectations, while budget properties may tolerate more risk, constrained by local alternative capacity.



---

## Submission Checklist

Before submitting, verify:

- [ ] All code cells run **top-to-bottom** without errors (Kernel → Restart & Run All)
- [ ] All tables and plots are visible in the notebook output
- [ ] All written-answer cells are filled in
- [ ] You have not modified the data-loading cell or the business parameters
- [ ] Your improved model does not use `reservation_status` or other leakage features
- [ ] You have defined `model_A`, `model_B`, `model_C`, `model_D` and they all have `.predict_proba()` methods
- [ ] You have defined `X_train_improved`, `X_test_improved`, and `probs_dict`

**Submit:** This completed notebook (.ipynb file)

---

## Grading Rubric

| Part | Points | Key Criteria |
|------|--------|-------------|
| 1. Data & Baseline | 15 | Correct metrics, calibration diagnosis |
| 2. Model Improvement | 25 | Justified feature engineering, tuning, calibration; four configs ready |
| 3. Decision Rule | 20 | Correct stopping logic, MC implementation, queue-based framing |
| 4. Policy Evaluation | 25 | Correct bootstrap, complete comparison, dollar-value decomposition |
| 4e. Normal Approximation | 5 (bonus) | Correct implementation, timing comparison, profit difference analysis |
| 5. Reflection | 15 | Thoughtful analysis connecting prediction to decision quality |
| **Total** | **100 + 5 bonus** | |